# Rebuild UCF101 sequences with a recording-group-aware split

**This is the replacement notebook for the sequence-building/split stage.** It does not run YOLO, ViTPose, or extract keypoints again. It reads the completed `.npz` files that already exist in `data/processed/keypoints_batch_all/`, assigns entire UCF101 recording groups to one split, and rebuilds the LSTM arrays in a **new output folder**.

### What this fixes

The previous builder split individual videos randomly within each class. That prevents the *same video* from crossing splits, but can place clips from the same UCF101 recording group (`..._g12_c01.avi`, `..._g12_c02.avi`) in both training and test. This notebook splits by **class + recording group** so that all clips in a parsed group stay together.

### Safety

- Existing `keypoints_batch_all/`, `sequences_all/`, model checkpoints and reports are not modified.
- New sequence arrays are written to `data/processed/sequences_group_split_v1/`.
- Default split target is 60% train / 20% validation / 20% test by video count. Because groups must stay intact, exact percentages may differ.
- This is a **group-aware randomized split**, not one of the official UCF101 split-list benchmarks. For a paper-comparable UCF101 result, later evaluate the official split lists separately.
- If a filename does not contain a UCF101-style `_gNN_` group marker, the notebook flags it; those videos need manual inspection because a reliable recording-group split cannot be inferred from the filename alone. A class with fewer than three recording groups cannot be represented in all three splits without leakage; such classes are reported separately.

Run all cells from top to bottom. It should take substantially less time than pose extraction because it only reads saved keypoints.

In [1]:
from pathlib import Path
import re
import json
import gc
import hashlib
import itertools
import warnings
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
KEYPOINT_DIR = PROCESSED_DIR / "keypoints_batch_all"
MANIFEST_PATH = PROCESSED_DIR / "manifest.csv"

# IMPORTANT: new destination; does not overwrite sequences_all/
SEQUENCE_DIR = PROCESSED_DIR / "sequences_group_split_v1"
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

SEQUENCE_LENGTH = 16
SEQUENCE_STRIDE = 4
MIN_VALID_FRAME_RATIO = 0.80
TRAIN_FRACTION = 0.60
VAL_FRACTION = 0.20
TEST_FRACTION = 0.20
SPLIT_SEED = 42

assert KEYPOINT_DIR.is_dir(), f"Completed keypoint directory not found: {KEYPOINT_DIR}"
assert abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) < 1e-8

print("Project:", PROJECT_DIR)
print("Input keypoints:", KEYPOINT_DIR)
print("Output sequences:", SEQUENCE_DIR)
print("Sequence length/stride:", SEQUENCE_LENGTH, SEQUENCE_STRIDE)
print("Target split:", TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION)


Project: C:\Users\admin\ViTPose_Action_Recognition
Input keypoints: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_batch_all
Output sequences: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_group_split_v1
Sequence length/stride: 16 4
Target split: 0.6 0.2 0.2


## 1. Scan completed per-video keypoint files

This stage reads only metadata first, so it does not load all pose arrays into RAM.

In [2]:
complete_npz_files = sorted(
    p for p in KEYPOINT_DIR.glob("*.npz")
    if ".partial." not in p.name and ".tmp." not in p.name
)
print("Completed NPZ files found:", len(complete_npz_files))
assert complete_npz_files, "No completed NPZ files found. Check KEYPOINT_DIR."

video_meta = []
bad_files = []
for file_index, npz_path in enumerate(complete_npz_files, start=1):
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            required = {"keypoints", "boxes", "frame_indices", "has_person", "label", "video_path"}
            missing = required - set(d.files)
            if missing:
                raise ValueError(f"Missing NPZ keys: {sorted(missing)}")
            label = str(d["label"].item())
            video_path = str(d["video_path"].item())
            n_samples = int(len(d["frame_indices"]))
            n_pose_ok = int(np.asarray(d["has_person"], dtype=bool).sum())
        video_meta.append({
            "npz_path": str(npz_path),
            "video_path": video_path,
            "label": label,
            "n_samples": n_samples,
            "n_pose_ok": n_pose_ok,
        })
    except Exception as exc:
        bad_files.append({"npz_path": str(npz_path), "error": repr(exc)})
    if file_index % 1000 == 0:
        print(f"Scanned {file_index}/{len(complete_npz_files)} NPZ metadata files...", flush=True)

if bad_files:
    bad_path = SEQUENCE_DIR / "unreadable_keypoint_files.csv"
    pd.DataFrame(bad_files).to_csv(bad_path, index=False)
    print(f"WARNING: {len(bad_files)} unreadable NPZ files. Details: {bad_path}")

video_meta_df = pd.DataFrame(video_meta)
assert not video_meta_df.empty, "No readable keypoint metadata found."
# If duplicates exist, keep one record per source video; report the number removed.
duplicate_count = int(video_meta_df.duplicated(subset=["video_path"]).sum())
if duplicate_count:
    print("Duplicate source-video NPZ metadata rows removed:", duplicate_count)
video_meta_df = video_meta_df.sort_values(["label", "video_path", "npz_path"]).drop_duplicates(
    subset=["video_path"], keep="first"
).reset_index(drop=True)

print("Unique readable source videos:", len(video_meta_df))
print("Classes:", video_meta_df["label"].nunique())
print("Videos per class (lowest counts):")
display(video_meta_df.groupby("label").size().sort_values().head(15).rename("videos").reset_index())
print("Videos per class (highest counts):")
display(video_meta_df.groupby("label").size().sort_values(ascending=False).head(15).rename("videos").reset_index())

# Optional completeness comparison against the project's manifest.
if MANIFEST_PATH.is_file():
    try:
        source_manifest = pd.read_csv(MANIFEST_PATH)
        if {"video_path", "label"}.issubset(source_manifest.columns):
            expected_paths = set(source_manifest["video_path"].astype(str))
            found_paths = set(video_meta_df["video_path"].astype(str))
            missing_npz_paths = sorted(expected_paths - found_paths)
            unexpected_npz_paths = sorted(found_paths - expected_paths)
            print("Manifest video rows:", len(expected_paths))
            print("Manifest videos without readable complete NPZ:", len(missing_npz_paths))
            if missing_npz_paths:
                pd.DataFrame({"video_path": missing_npz_paths}).to_csv(
                    SEQUENCE_DIR / "manifest_videos_without_keypoints.csv", index=False
                )
            if unexpected_npz_paths:
                pd.DataFrame({"video_path": unexpected_npz_paths}).to_csv(
                    SEQUENCE_DIR / "keypoints_not_in_manifest.csv", index=False
                )
    except Exception as exc:
        print("Manifest comparison skipped due to:", repr(exc))


Completed NPZ files found: 13319
Scanned 1000/13319 NPZ metadata files...
Scanned 2000/13319 NPZ metadata files...
Scanned 3000/13319 NPZ metadata files...
Scanned 4000/13319 NPZ metadata files...
Scanned 5000/13319 NPZ metadata files...
Scanned 6000/13319 NPZ metadata files...
Scanned 7000/13319 NPZ metadata files...
Scanned 8000/13319 NPZ metadata files...
Scanned 9000/13319 NPZ metadata files...
Scanned 10000/13319 NPZ metadata files...
Scanned 11000/13319 NPZ metadata files...
Scanned 12000/13319 NPZ metadata files...
Scanned 13000/13319 NPZ metadata files...
Unique readable source videos: 13319
Classes: 101
Videos per class (lowest counts):


,label,videos
0,Skijet,100
1,TaiChi,100
2,PullUps,100
3,PlayingViolin,100
4,BreastStroke,101
5,PushUps,102
6,UnevenBars,104
7,PlayingPiano,105
8,BalanceBeam,108
9,BlowingCandles,109


Videos per class (highest counts):


,label,videos
0,CricketShot,167
1,TennisSwing,166
2,PlayingDhol,164
3,HorseRiding,164
4,PlayingCello,164
5,BoxingPunchingBag,163
6,Drumming,161
7,ShavingBeard,161
8,Punch,160
9,BenchPress,160


Manifest video rows: 13320
Manifest videos without readable complete NPZ: 1


## 2. Extract recording groups and create a group-aware split

For standard UCF101 filenames such as `v_Biking_g12_c03.avi`, the group is `label + g12`. The `cNN` clip number is not used to separate related clips. Whole groups are assigned to one split.

In [3]:
GROUP_PATTERN = re.compile(r"_g(\d+)_", flags=re.IGNORECASE)

def get_recording_group(video_path, label):
    stem = Path(str(video_path)).stem
    match = GROUP_PATTERN.search(stem)
    if match:
        group_id = f"g{int(match.group(1)):02d}"
        return f"{label}::{group_id}", group_id, True
    # Do not silently merge unparseable clips: mark each as a unique, unverified group.
    fallback = hashlib.sha1(str(video_path).encode("utf-8")).hexdigest()[:12]
    return f"{label}::UNPARSED::{fallback}", "UNPARSED", False

group_info = video_meta_df.apply(
    lambda r: get_recording_group(r["video_path"], r["label"]), axis=1
)
video_meta_df["recording_group_key"] = [x[0] for x in group_info]
video_meta_df["recording_group"] = [x[1] for x in group_info]
video_meta_df["group_parsed_from_filename"] = [x[2] for x in group_info]

unparsed_groups = video_meta_df.loc[~video_meta_df["group_parsed_from_filename"]].copy()
print("Videos with parsed UCF101 recording-group IDs:", int(video_meta_df["group_parsed_from_filename"].sum()))
print("Videos without a parseable _gNN_ marker:", len(unparsed_groups))
if len(unparsed_groups):
    unparsed_path = SEQUENCE_DIR / "unparsed_recording_groups.csv"
    unparsed_groups[["video_path", "label", "npz_path"]].to_csv(unparsed_path, index=False)
    print("WARNING: group isolation cannot be verified for these filenames.")
    print("Inspect:", unparsed_path)

SPLIT_NAMES = ["train", "val", "test"]
SPLIT_FRACTIONS = np.array([TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION], dtype=np.float64)

def assign_groups_for_class(class_df, seed):
    """Assign indivisible recording groups to splits, approximately matching video fractions."""
    group_records = []
    for group_key, gdf in class_df.groupby("recording_group_key", sort=True):
        group_records.append({
            "recording_group_key": str(group_key),
            "recording_group": str(gdf["recording_group"].iloc[0]),
            "video_paths": sorted(gdf["video_path"].astype(str).tolist()),
            "size": int(len(gdf)),
        })

    rng = np.random.RandomState(seed)
    # Random tie-breaking followed by largest groups first gives reproducible balancing.
    rng.shuffle(group_records)
    group_records.sort(key=lambda g: g["size"], reverse=True)
    n_groups = len(group_records)
    n_videos = int(len(class_df))
    targets = SPLIT_FRACTIONS * n_videos
    current = np.zeros(3, dtype=np.float64)
    assignments = {}

    if n_groups >= 3:
        # Allocate the first three groups one per split using the best of six permutations.
        first_three = group_records[:3]
        best_perm, best_cost = None, float("inf")
        for perm in itertools.permutations(range(3)):
            trial = np.zeros(3, dtype=np.float64)
            for rec, split_idx in zip(first_three, perm):
                trial[split_idx] += rec["size"]
            cost = np.sum(((trial - targets) / np.maximum(targets, 1.0)) ** 2)
            if cost < best_cost:
                best_cost, best_perm = cost, perm
        for rec, split_idx in zip(first_three, best_perm):
            assignments[rec["recording_group_key"]] = SPLIT_NAMES[split_idx]
            current[split_idx] += rec["size"]
        remaining = group_records[3:]
    elif n_groups == 2:
        # With only two groups, it is impossible to populate all 3 splits without
        # leakage. Keep one group in train and the other in test; validation lacks
        # this class and the final audit will make this limitation visible.
        assignments[group_records[0]["recording_group_key"]] = "train"
        assignments[group_records[1]["recording_group_key"]] = "test"
        current[0] += group_records[0]["size"]
        current[2] += group_records[1]["size"]
        remaining = []
    elif n_groups == 1:
        assignments[group_records[0]["recording_group_key"]] = "train"
        current[0] += group_records[0]["size"]
        remaining = []
    else:
        remaining = []

    for rec in remaining:
        costs = []
        for split_idx in range(3):
            trial = current.copy()
            trial[split_idx] += rec["size"]
            cost = np.sum(((trial - targets) / np.maximum(targets, 1.0)) ** 2)
            costs.append(cost)
        chosen = int(np.argmin(costs))
        assignments[rec["recording_group_key"]] = SPLIT_NAMES[chosen]
        current[chosen] += rec["size"]

    # For classes with 2 groups, ensure the first group goes to train, second to test
    # only if the greedy allocation put all groups in a single split (normally it won't).
    return assignments, n_groups

split_rows = []
low_group_classes = []
for class_offset, (label, class_df) in enumerate(video_meta_df.groupby("label", sort=True)):
    assignments, n_groups = assign_groups_for_class(
        class_df, SPLIT_SEED + class_offset * 1009
    )
    if n_groups < 3:
        low_group_classes.append({"label": label, "recording_groups": n_groups, "videos": len(class_df)})
    local = class_df.copy()
    local["split"] = local["recording_group_key"].map(assignments)
    if local["split"].isna().any():
        raise RuntimeError(f"Some videos in class {label!r} were not assigned to a split.")
    split_rows.append(local)

video_split_df = pd.concat(split_rows, ignore_index=True)
video_split_df = video_split_df.sort_values(["label", "video_path"]).reset_index(drop=True)
video_split_df["video_id"] = np.arange(len(video_split_df), dtype=np.int32)

if low_group_classes:
    low_path = SEQUENCE_DIR / "classes_with_fewer_than_3_groups.csv"
    pd.DataFrame(low_group_classes).to_csv(low_path, index=False)
    print("WARNING: some classes have fewer than 3 groups; see", low_path)

# Hard checks: each group key and source video must occur in exactly one split.
assert video_split_df.groupby("recording_group_key")["split"].nunique().max() == 1, \
    "Recording-group leakage detected: a group was assigned to multiple splits."
assert video_split_df.groupby("video_path")["split"].nunique().max() == 1

video_id_map = dict(zip(video_split_df["video_path"].astype(str), video_split_df["video_id"].astype(int)))
video_split = dict(zip(video_split_df["video_path"].astype(str), video_split_df["split"].astype(str)))
video_split_df.to_csv(SEQUENCE_DIR / "video_split_manifest.csv", index=False)

group_split_df = video_split_df[["label", "recording_group_key", "recording_group", "split"]].drop_duplicates()
group_split_df.to_csv(SEQUENCE_DIR / "group_split_manifest.csv", index=False)

print("\nVideo counts by split:")
display(video_split_df.groupby("split").size().reindex(SPLIT_NAMES).rename("videos").reset_index())
print("Video counts by class and split:")
display(video_split_df.groupby(["label", "split"]).size().unstack(fill_value=0).reindex(columns=SPLIT_NAMES))
print("Unique recording groups by split:")
display(group_split_df.groupby("split").size().reindex(SPLIT_NAMES).rename("recording_groups").reset_index())
print("Saved split manifest:", SEQUENCE_DIR / "video_split_manifest.csv")
print("Saved group manifest:", SEQUENCE_DIR / "group_split_manifest.csv")


Videos with parsed UCF101 recording-group IDs: 13319
Videos without a parseable _gNN_ marker: 0

Video counts by split:


,split,videos
0,train,7999
1,val,2654
2,test,2666


Video counts by class and split:


split,train,val,test
label,,,
ApplyEyeMakeup,88,27,30
ApplyLipstick,69,21,24
Archery,87,31,27
BabyCrawling,76,28,28
BalanceBeam,65,20,23
...,...,...,...
VolleyballSpiking,71,23,22
WalkingWithDog,74,23,26
WallPushups,77,25,28


Unique recording groups by split:


,split,recording_groups
0,train,1614
1,val,450
2,test,461


Saved split manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_group_split_v1\video_split_manifest.csv
Saved group manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_group_split_v1\group_split_manifest.csv


## 3. Convert per-video poses into normalized fixed-length windows

This keeps the previous sequence recipe so that the main change is the split protocol, not a simultaneous preprocessing change.

In [4]:
def normalize_video_poses(keypoints, boxes, pose_ok):
    """Use the existing person-box-normalized x/y and confidence-like score format."""
    T = len(keypoints)
    normalized = np.zeros((T, 17, 3), dtype=np.float32)
    valid_frame = np.zeros(T, dtype=bool)
    for t in range(T):
        kp, box = keypoints[t], boxes[t]
        if not pose_ok[t] or not np.isfinite(box).all() or not np.isfinite(kp[:, :2]).all():
            continue
        x1, y1, x2, y2 = map(float, box)
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue
        normalized[t, :, 0] = (kp[:, 0] - x1) / bw
        normalized[t, :, 1] = (kp[:, 1] - y1) / bh
        scores = kp[:, 2].copy()
        scores[~np.isfinite(scores)] = 0.0
        normalized[t, :, 2] = np.clip(scores, 0.0, 1.0)
        normalized[t] = np.nan_to_num(normalized[t], nan=0.0, posinf=0.0, neginf=0.0)
        valid_frame[t] = True
    return normalized, valid_frame


def iter_windows(npz_path):
    with np.load(npz_path, allow_pickle=False) as d:
        kps = np.asarray(d["keypoints"], dtype=np.float32)
        boxes = np.asarray(d["boxes"], dtype=np.float32)
        frame_indices = np.asarray(d["frame_indices"], dtype=np.int32)
        pose_ok = np.asarray(d["has_person"], dtype=bool)
    if kps.ndim != 3 or kps.shape[1:] != (17, 3):
        return
    if not (len(kps) == len(boxes) == len(frame_indices) == len(pose_ok)):
        return
    normalized, valid_frame = normalize_video_poses(kps, boxes, pose_ok)
    T = len(normalized)
    if T < SEQUENCE_LENGTH:
        return
    starts = list(range(0, T - SEQUENCE_LENGTH + 1, SEQUENCE_STRIDE))
    final_start = T - SEQUENCE_LENGTH
    if starts and starts[-1] != final_start:
        starts.append(final_start)
    for start in starts:
        end = start + SEQUENCE_LENGTH
        valid_ratio = float(valid_frame[start:end].mean())
        if valid_ratio < MIN_VALID_FRAME_RATIO:
            continue
        window = np.nan_to_num(normalized[start:end], nan=0.0, posinf=0.0, neginf=0.0)
        yield window.reshape(SEQUENCE_LENGTH, 51), int(frame_indices[start]), int(frame_indices[end - 1]), valid_ratio

print("Window generation functions ready.")
print("Settings: length=", SEQUENCE_LENGTH, "stride=", SEQUENCE_STRIDE,
      "minimum valid-frame ratio=", MIN_VALID_FRAME_RATIO)


Window generation functions ready.
Settings: length= 16 stride= 4 minimum valid-frame ratio= 0.8


## 4. Count windows per split before allocating output arrays

First pass counts the number of usable windows per video. This lets the builder create exact-size disk-backed arrays without loading all windows into RAM.

In [5]:
video_meta_records = video_split_df.to_dict("records")
counts = {"train": 0, "val": 0, "test": 0}
per_video_counts = {}
per_video_quality = {}

for i, row in enumerate(video_meta_records, start=1):
    npz_path = row["npz_path"]
    video_path = str(row["video_path"])
    split = str(row["split"])
    try:
        n_windows = sum(1 for _ in iter_windows(npz_path))
    except Exception as exc:
        print("ERROR reading pose windows for", Path(npz_path).name, repr(exc))
        n_windows = 0
    per_video_counts[video_path] = int(n_windows)
    counts[split] += int(n_windows)
    per_video_quality[video_path] = {
        "sampled_frames": int(row["n_samples"]),
        "pose_successes": int(row["n_pose_ok"]),
        "sequence_windows": int(n_windows),
    }
    if i % 1000 == 0:
        print(f"Counted windows for {i}/{len(video_meta_records)} videos...", flush=True)

print("\nSequence windows to write:", counts)
print("Videos with at least one usable window:", sum(n > 0 for n in per_video_counts.values()), "/", len(per_video_counts))
assert counts["train"] > 0 and counts["val"] > 0 and counts["test"] > 0, \
    "One split has zero usable windows. Check keypoint completeness or sequence settings."

all_labels = set(video_split_df["label"].astype(str))
trainable_labels = {
    str(row["label"]) for row in video_meta_records
    if row["split"] == "train" and per_video_counts.get(str(row["video_path"]), 0) > 0
}
missing_train_labels = sorted(all_labels - trainable_labels)
if missing_train_labels:
    raise ValueError(
        "These classes have no usable training sequences under the group split: "
        + ", ".join(missing_train_labels)
        + ". Inspect extraction quality and group counts before training."
    )


Counted windows for 1000/13319 videos...
Counted windows for 2000/13319 videos...
Counted windows for 3000/13319 videos...
Counted windows for 4000/13319 videos...
Counted windows for 5000/13319 videos...
Counted windows for 6000/13319 videos...
Counted windows for 7000/13319 videos...
Counted windows for 8000/13319 videos...
Counted windows for 9000/13319 videos...
Counted windows for 10000/13319 videos...
Counted windows for 11000/13319 videos...
Counted windows for 12000/13319 videos...
Counted windows for 13000/13319 videos...

Sequence windows to write: {'train': 61747, 'val': 21815, 'test': 21960}
Videos with at least one usable window: 12125 / 13319


## 5. Write train/validation/test arrays using memory maps

The saved structure stays compatible with the LSTM and video-level aggregation notebooks: `X_*.npy`, `y_*.npy`, `videos_*.npy`, and `label_map.json`. The output folder is new, so the old `sequences_all/` files remain intact.

In [6]:
label_names = sorted(video_split_df["label"].astype(str).unique().tolist())
label_to_id = {name: i for i, name in enumerate(label_names)}
label_map = {str(i): name for name, i in label_to_id.items()}
(SEQUENCE_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")

array_paths = {}
memmaps = {}
for split in SPLIT_NAMES:
    array_paths[split] = {
        "X": SEQUENCE_DIR / f"X_{split}.npy",
        "y": SEQUENCE_DIR / f"y_{split}.npy",
        "videos": SEQUENCE_DIR / f"videos_{split}.npy",
    }
    memmaps[split] = {
        "X": np.lib.format.open_memmap(
            array_paths[split]["X"], mode="w+", dtype=np.float32,
            shape=(counts[split], SEQUENCE_LENGTH, 51)
        ),
        "y": np.lib.format.open_memmap(
            array_paths[split]["y"], mode="w+", dtype=np.int64,
            shape=(counts[split],)
        ),
        "videos": np.lib.format.open_memmap(
            array_paths[split]["videos"], mode="w+", dtype=np.int32,
            shape=(counts[split],)
        ),
    }

offsets = {split: 0 for split in SPLIT_NAMES}
sequence_rows = []

for i, row in enumerate(video_meta_records, start=1):
    vp = str(row["video_path"])
    label = str(row["label"])
    split = str(row["split"])
    npz_path = str(row["npz_path"])
    start_offset = offsets[split]
    video_id = int(row["video_id"])
    try:
        for window, start_frame, end_frame, valid_ratio in iter_windows(npz_path):
            idx = offsets[split]
            memmaps[split]["X"][idx] = window
            memmaps[split]["y"][idx] = label_to_id[label]
            memmaps[split]["videos"][idx] = video_id
            offsets[split] += 1
    except Exception as exc:
        raise RuntimeError(f"Failed while writing windows for {vp}: {exc!r}") from exc

    sequence_rows.append({
        "video_id": video_id,
        "video_path": vp,
        "label": label,
        "recording_group_key": str(row["recording_group_key"]),
        "recording_group": str(row["recording_group"]),
        "group_parsed_from_filename": bool(row["group_parsed_from_filename"]),
        "split": split,
        "sequence_windows": int(offsets[split] - start_offset),
        "sampled_frames": int(row["n_samples"]),
        "pose_successes": int(row["n_pose_ok"]),
        "source_npz": npz_path,
    })
    if i % 500 == 0:
        print(f"Wrote windows for {i}/{len(video_meta_records)} videos...", flush=True)

for split in SPLIT_NAMES:
    for arr in memmaps[split].values():
        arr.flush()
    if offsets[split] != counts[split]:
        raise RuntimeError(f"Count mismatch for {split}: wrote {offsets[split]}, expected {counts[split]}")
    print(f"Saved {split}: X={memmaps[split]['X'].shape}, y={memmaps[split]['y'].shape}, videos={len(np.unique(memmaps[split]['videos']))}")

del memmaps
gc.collect()

sequence_manifest = pd.DataFrame(sequence_rows)
sequence_manifest_path = SEQUENCE_DIR / "video_sequence_summary.csv"
sequence_manifest.to_csv(sequence_manifest_path, index=False)
print("Saved per-video sequence summary:", sequence_manifest_path)


Wrote windows for 500/13319 videos...
Wrote windows for 1000/13319 videos...
Wrote windows for 1500/13319 videos...
Wrote windows for 2000/13319 videos...
Wrote windows for 2500/13319 videos...
Wrote windows for 3000/13319 videos...
Wrote windows for 3500/13319 videos...
Wrote windows for 4000/13319 videos...
Wrote windows for 4500/13319 videos...
Wrote windows for 5000/13319 videos...
Wrote windows for 5500/13319 videos...
Wrote windows for 6000/13319 videos...
Wrote windows for 6500/13319 videos...
Wrote windows for 7000/13319 videos...
Wrote windows for 7500/13319 videos...
Wrote windows for 8000/13319 videos...
Wrote windows for 8500/13319 videos...
Wrote windows for 9000/13319 videos...
Wrote windows for 9500/13319 videos...
Wrote windows for 10000/13319 videos...
Wrote windows for 10500/13319 videos...
Wrote windows for 11000/13319 videos...
Wrote windows for 11500/13319 videos...
Wrote windows for 12000/13319 videos...
Wrote windows for 12500/13319 videos...
Wrote windows for 13

## 6. Final split and dataset checks

This verifies that videos and parsed recording groups do not cross splits, confirms the saved array shapes, and writes a metadata JSON with the exact split settings.

In [7]:
# Check source videos and recording groups remain isolated.
video_sets = {
    split: set(video_split_df.loc[video_split_df["split"].eq(split), "video_path"].astype(str))
    for split in SPLIT_NAMES
}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not (video_sets[a] & video_sets[b]), f"Source-video overlap between {a} and {b}."

group_sets = {
    split: set(video_split_df.loc[video_split_df["split"].eq(split), "recording_group_key"].astype(str))
    for split in SPLIT_NAMES
}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = group_sets[a] & group_sets[b]
    assert not overlap, f"Recording-group leakage between {a} and {b}: {list(overlap)[:5]}"

print("DATASET SUMMARY")
print("Protocol: class-aware UCF101 recording-group split (randomized, not official split lists)")
print("Classes:", len(label_map))
print("Source videos with readable keypoints:", len(video_split_df))
print("Unique parsed+fallback group keys:", video_split_df["recording_group_key"].nunique())
print("Unparsed group filenames:", int((~video_split_df["group_parsed_from_filename"]).sum()))
print("Sequence length:", SEQUENCE_LENGTH, "| stride:", SEQUENCE_STRIDE, "| features/frame: 51")

for split in SPLIT_NAMES:
    X = np.load(SEQUENCE_DIR / f"X_{split}.npy", mmap_mode="r", allow_pickle=False)
    y = np.load(SEQUENCE_DIR / f"y_{split}.npy", mmap_mode="r", allow_pickle=False)
    vids = np.load(SEQUENCE_DIR / f"videos_{split}.npy", mmap_mode="r", allow_pickle=False)
    print(f"{split}: X={X.shape}, y={y.shape}, unique source videos={len(np.unique(vids))}")
    print("  window counts by class:", {
        label_map[str(int(k))]: int(v)
        for k, v in zip(*np.unique(y, return_counts=True))
    })
    del X, y, vids
    gc.collect()

metadata = {
    "dataset": "UCF101 pose sequences",
    "split_protocol": "group-aware randomized split within class; not official UCF101 split lists",
    "fractions_target": {"train": TRAIN_FRACTION, "val": VAL_FRACTION, "test": TEST_FRACTION},
    "seed": SPLIT_SEED,
    "sequence_length": SEQUENCE_LENGTH,
    "sequence_stride": SEQUENCE_STRIDE,
    "min_valid_frame_ratio": MIN_VALID_FRAME_RATIO,
    "features_per_frame": 51,
    "num_classes": int(len(label_map)),
    "readable_source_videos": int(len(video_split_df)),
    "source_npz_directory": str(KEYPOINT_DIR),
    "output_directory": str(SEQUENCE_DIR),
    "unparsed_group_filenames": int((~video_split_df["group_parsed_from_filename"]).sum()),
    "windows_by_split": {k: int(v) for k, v in counts.items()},
}
metadata_path = SEQUENCE_DIR / "dataset_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("\nPASS: no source-video overlap and no recording-group-key overlap across splits.")
print("Metadata:", metadata_path)
print("NEW SEQUENCE DIRECTORY:", SEQUENCE_DIR.resolve())
print("Use this path in the LSTM notebook: data/processed/sequences_group_split_v1")


DATASET SUMMARY
Protocol: class-aware UCF101 recording-group split (randomized, not official split lists)
Classes: 101
Source videos with readable keypoints: 13319
Unique parsed+fallback group keys: 2525
Unparsed group filenames: 0
Sequence length: 16 | stride: 4 | features/frame: 51
train: X=(61747, 16, 51), y=(61747,), unique source videos=7236
  window counts by class: {'ApplyEyeMakeup': 688, 'ApplyLipstick': 596, 'Archery': 757, 'BabyCrawling': 577, 'BalanceBeam': 258, 'BandMarching': 979, 'BaseballPitch': 351, 'Basketball': 426, 'BasketballDunk': 149, 'BenchPress': 412, 'Biking': 749, 'Billiards': 1056, 'BlowDryHair': 966, 'BlowingCandles': 226, 'BodyWeightSquats': 359, 'Bowling': 423, 'BoxingPunchingBag': 871, 'BoxingSpeedBag': 892, 'BreastStroke': 19, 'BrushingTeeth': 813, 'CleanAndJerk': 682, 'CliffDiving': 65, 'CricketBowling': 297, 'CricketShot': 289, 'CuttingInKitchen': 483, 'Diving': 385, 'Drumming': 1026, 'Fencing': 350, 'FieldHockeyPenalty': 350, 'FloorGymnastics': 450, '

## Important after this notebook finishes

1. Open your LSTM training notebook, make a copy, and change its sequence-directory setting from `sequences_all` to `sequences_group_split_v1`.
2. Train from scratch on the new train split. Do not reuse the old trained checkpoint as if it were trained on this new split.
3. Keep the new model's filenames separate from old models.
4. Compare future experiments using this same group-aware split. For official benchmark results, implement the official UCF101 split-list protocol as a separate dataset version.

Expected files in the new folder: `X_train.npy`, `y_train.npy`, `videos_train.npy`, corresponding `val` and `test` arrays, `label_map.json`, `video_split_manifest.csv`, `group_split_manifest.csv`, `video_sequence_summary.csv`, and `dataset_metadata.json`.